# Overborrowing and Systemic Externalities in the Business Cycle Under Imperfect Information

## Steps 0 & 1: Parameters and Transition Matrix (Perfect Information)

This notebook creates the parameter structure and discretizes the exogenous stochastic processes
for the **perfect information** version of the model.

- **Authors:** Juan Herreño (jherrenolopera@ucsd.edu) & Carlos Rondón Moreno (crondon@bcentral.cl)
- **Date:** 2025

### Contents
1. Import packages
2. Define the `Params` namedtuple and factory function
3. Display Table 2 from the paper
4. Discretize VAR(1) process using QuantEcon
5. Build perfect information transition matrix

In [38]:
# Import packages
import warnings
warnings.filterwarnings("ignore")

import jax
import jax.numpy as jnp
import numpy as np
from collections import namedtuple
import quantecon as qe
from quantecon.markov.approximation import discrete_var

jax.config.update("jax_enable_x64", True)

## Step 0: Parameters

In [39]:
Params = namedtuple(
    "Params",
    (
        # Simulation control
        "nodes",        # Nodes in state-space for exogenous variables
        "grid_points",  # Grid points for debt
        "burn",         # Burn-in period
        "Tsim",         # Total simulation horizon (including burn-in)
        # IRF control
        "horizn",       # IRF horizon
        "init_debt",    # Initial debt index for IRFs (steady state)
        # Permanent component
        "rho_g",        # Autocorrelation of permanent growth
        "sigmag",       # Std. dev. of permanent growth rate
        "g",            # Mean growth rate (Garcia-Cicco et al. 2010)
        # VAR(1) for endowments
        "A",            # Autocorrelation matrix (3x3)
        "Sigma",        # Variance-covariance matrix (3x3)
        # Deep parameters
        "beta",         # Discount factor
        "kappa",        # Borrowing constraint constant
        "r",            # Annual interest rate
        "rho",          # Risk aversion coefficient
        "omega",        # Weight of tradables in CES
        "eta",          # Intratemporal elasticity of substitution
        # Grid dimensions
        "ytn",          # Grid points: transitory tradable
        "ynn",          # Grid points: transitory non-tradable
        "gn",           # Grid points: permanent component
        "gTn",          # Grid points: growth rate tradable output
        "gNn",          # Grid points: growth rate non-tradable output
        # Debt grid
        "bmin",         # Minimum debt
        "bmax",         # Maximum debt
        "bn",           # Number of debt grid points
        "b",            # Debt grid (jnp array)
        # Other
        "nstd",         # Std. devs. for crisis definition
        "window",       # Years around crises for event study
    )
)


def create_params(
    # Simulation control
    nodes: int = 10,
    grid_points: int = 101,
    burn: int = 1_000_000,
    # IRF control
    horizn: int = 21,
    # Permanent component parameters
    rho_g: float = 0.496779121757470,
    sigmag: float = 0.00327510105056510,
    g: float = None,  # defaults to log(1.0131)
    # Deep parameters
    beta: float = 0.83,
    kappa: float = 0.335,
    r: float = 0.04,
    rho: float = 2.0,
    omega: float = 0.31,
    eta: float = 0.83,
    # Debt grid bounds
    bmin: float = -1.4,
    bmax: float = -0.2,
    # Crisis parameters
    nstd: float = 1.0,
    window: int = 5,
) -> Params:
    """
    Create the full parameter structure for the overborrowing model.

    Returns a Params namedtuple with all calibrated values, grids,
    and VAR(1) matrices.
    """

    # Derived quantities
    if g is None:
        g = float(jnp.log(1.0131))  # Garcia-Cicco et al. (2010)

    Tsim = burn + 1_000_000       # Total simulation length
    init_debt = grid_points // 2  # IRFs start at steady state
    bn = grid_points

    # Grid dimensions (all equal to nodes)
    ytn = ynn = gn = gTn = gNn = nodes

    # VAR(1) autocorrelation matrix
    A = jnp.array([
        [0.734679129925539, -0.255320869154147, 0.0],
        [0.033652507003993,  0.417047371095781, 0.0],
        [0.0,                0.0,               rho_g],
    ])

    # VAR(1) variance-covariance matrix
    Sigma = jnp.array([
        [0.00461817366335938, 0.000379607811872954, 0.0],
        [0.000379607811872954, 0.00137117483970497, 0.0],
        [0.0,                  0.0,                  sigmag],
    ])

    # Debt grid
    b = jnp.linspace(bmin, bmax, bn)

    return Params(
        nodes=nodes, grid_points=grid_points, burn=burn, Tsim=Tsim,
        horizn=horizn, init_debt=init_debt,
        rho_g=rho_g, sigmag=sigmag, g=g,
        A=A, Sigma=Sigma,
        beta=beta, kappa=kappa, r=r, rho=rho, omega=omega, eta=eta,
        ytn=ytn, ynn=ynn, gn=gn, gTn=gTn, gNn=gNn,
        bmin=bmin, bmax=bmax, bn=bn, b=b,
        nstd=nstd, window=window,
    )

In [40]:
def print_params(params):
    """Print all parameters of a Params namedtuple."""
    print("Model Parameters:")
    print("=" * 60)
    for field_name in Params._fields:
        value = getattr(params, field_name)
        print(f"  {field_name:15s}  ->  {value}")
    print("=" * 60)


params = create_params()
print_params(params)

Model Parameters:
  nodes            ->  10
  grid_points      ->  101
  burn             ->  1000000
  Tsim             ->  2000000
  horizn           ->  21
  init_debt        ->  50
  rho_g            ->  0.49677912175747
  sigmag           ->  0.0032751010505651
  g                ->  0.013014937077494544
  A                ->  [[ 0.73467913 -0.25532087  0.        ]
 [ 0.03365251  0.41704737  0.        ]
 [ 0.          0.          0.49677912]]
  Sigma            ->  [[0.00461817 0.00037961 0.        ]
 [0.00037961 0.00137117 0.        ]
 [0.         0.         0.0032751 ]]
  beta             ->  0.83
  kappa            ->  0.335
  r                ->  0.04
  rho              ->  2.0
  omega            ->  0.31
  eta              ->  0.83
  ytn              ->  10
  ynn              ->  10
  gn               ->  10
  gTn              ->  10
  gNn              ->  10
  bmin             ->  -1.4
  bmax             ->  -0.2
  bn               ->  101
  b                ->  [-1.4   -1.3

### Table 2: Calibrated Parameters

Following Bianchi (2011), with $\beta$ and $\kappa$ calibrated to match Argentine data.

In [41]:
def display_table2(params):
    """Reproduce Table 2 from the paper."""
    rows = [
        ("r",         params.r,     "Interest Rate"),
        ("σ",         params.rho,   "Inverse of IES"),
        ("ε",         params.eta,   "Elasticity tradable / non-tradable"),
        ("ω",         params.omega, "Weight of tradable goods"),
        ("β",         params.beta,  "Discount factor"),
        ("κ",         params.kappa, "Borrowing constraint constant"),
        ("μ_g",       params.g,     "Avg. growth rate"),
    ]
    print("+" + "-" * 14 + "+" + "-" * 12 + "+" + "-" * 42 + "+")
    print(f"| {'Parameter':<12s} | {'Value':>10s} | {'Meaning':<40s} |")
    print("+" + "-" * 14 + "+" + "-" * 12 + "+" + "-" * 42 + "+")
    for name, val, desc in rows:
        print(f"| {name:<12s} | {val:10.4f} | {desc:<40s} |")
    print("+" + "-" * 14 + "+" + "-" * 12 + "+" + "-" * 42 + "+")


display_table2(params)

+--------------+------------+------------------------------------------+
| Parameter    |      Value | Meaning                                  |
+--------------+------------+------------------------------------------+
| r            |     0.0400 | Interest Rate                            |
| σ            |     2.0000 | Inverse of IES                           |
| ε            |     0.8300 | Elasticity tradable / non-tradable       |
| ω            |     0.3100 | Weight of tradable goods                 |
| β            |     0.8300 | Discount factor                          |
| κ            |     0.3350 | Borrowing constraint constant            |
| μ_g          |     0.0130 | Avg. growth rate                         |
+--------------+------------+------------------------------------------+


## Step 1: Transition Matrix (Perfect Information)

Discretize the 3-variable VAR(1) process for $[z^T_t,\; z^N_t,\; g_t]$ using the
simulation-based method from Schmitt-Grohé and Uribe (2010), implemented via
QuantEcon's `discrete_var`.

The process is:
$$x_t = A\, x_{t-1} + C\, u_t, \qquad u_t \sim \mathcal{N}(0, I)$$

where $C$ is the Cholesky factor of the variance-covariance matrix $\Sigma$, so that
$C\,C^\top = \Sigma$.

In [52]:
print("Discretizing VAR(1) process ...")

# Volatility matrix: Cholesky factor of the variance-covariance matrix
A_np = np.array(params.A)
Sigma_np = np.array(params.Sigma)
C = np.linalg.cholesky(Sigma_np)

# Discretize the VAR(1) using QuantEcon (Schmitt-Grohé & Uribe 2010)
# This simulates the process and bins into a Cartesian grid.
# States that are never visited are automatically removed.
mc = discrete_var(
    A=A_np,
    C=C,
    grid_sizes=np.array([params.ytn, params.ynn, params.gn]),
    sim_length=params.Tsim,
    random_state=0,
)

# Simulate the discretized Markov chain (for use in later steps)
# Xsim: state values at each period, shape (Tsim, 3) — columns are [yt, yn, gt]
# Xsim_idx: index into S (the unique state grid) at each period, shape (Tsim,)
Xsim = mc.simulate(ts_length=params.Tsim, random_state=0)
Xsim_idx = mc.simulate_indices(ts_length=params.Tsim, random_state=0)

print(f"Done.")
print(f"  Transition matrix shape:  {mc.P.shape}")
print(f"  State-space size (after trimming unreachable states): {mc.state_values.shape[0]}")
print(f"  Simulation values shape:  {Xsim.shape}")
print(f"  Simulation indices shape: {Xsim_idx.shape}")

Discretizing VAR(1) process ...
Done.
  Transition matrix shape:  (941, 941)
  State-space size (after trimming unreachable states): 941
  Simulation values shape:  (2000000, 3)
  Simulation indices shape: (2000000,)


### Build Perfect Information Objects

Extract the grids and transition matrix, shift the permanent component by the mean growth rate $\mu_g$,
and convert everything to JAX arrays.

In [53]:
# Grid values (deviations from mean, centered at zero)
yt = jnp.array(np.unique(mc.state_values[:, 0]))  # Transitory tradable endowment
yn = jnp.array(np.unique(mc.state_values[:, 1]))  # Transitory non-tradable endowment
gt = jnp.array(np.unique(mc.state_values[:, 2]))  # Permanent component (deviations)

# State matrix: shift growth rate column by mean growth rate g
S = jnp.array(mc.state_values)
S = S.at[:, 2].add(params.g)

# Transition probability matrix
Pi = jnp.array(mc.P)

# Number of states in the irreducible state-space
Yn = S.shape[0]

print(f"Perfect information transition matrix created.")
print(f"  Number of states (Yn): {Yn}")
print(f"  Tradable grid (yt):     {yt.shape[0]} pts,  range [{yt[0]:.6f}, {yt[-1]:.6f}]")
print(f"  Non-tradable grid (yn): {yn.shape[0]} pts,  range [{yn[0]:.6f}, {yn[-1]:.6f}]")
print(f"  Growth grid (gt):       {gt.shape[0]} pts,  range [{gt[0]:.6f}, {gt[-1]:.6f}]")
print(f"  Pi shape: {Pi.shape},  S shape: {S.shape}")

Perfect information transition matrix created.
  Number of states (Yn): 941
  Tradable grid (yt):     10 pts,  range [-0.312454, 0.312454]
  Non-tradable grid (yn): 10 pts,  range [-0.130176, 0.130176]
  Growth grid (gt):       10 pts,  range [-0.208523, 0.208523]
  Pi shape: (941, 941),  S shape: (941, 3)


### Store Results

Pack the perfect information transition matrix into a `TranMatFI` namedtuple for downstream use
in Steps 2–6.

In [55]:
TranMatFI = namedtuple("TranMatFI", ("Pi", "S", "Yn", "yt", "yn", "gt", "Xsim", "Xsim_idx"))

# Store simulation as JAX arrays
# Xsim:     state values (Tsim, 3) — the realized [yt, yn, gt] at each period
# Xsim_idx: state indices (Tsim,)  — row index into S at each period
Xsim_jax = jnp.array(Xsim)
Xsim_idx_jax = jnp.array(Xsim_idx)

tranmat_fi = TranMatFI(Pi=Pi, S=S, Yn=Yn, yt=yt, yn=yn, gt=gt, Xsim=Xsim_jax, Xsim_idx=Xsim_idx_jax)

print("TranMatFI namedtuple created with fields:")
for field in TranMatFI._fields:
    val = getattr(tranmat_fi, field)
    if hasattr(val, 'shape'):
        print(f"  {field:10s}  ->  shape {val.shape}, dtype {val.dtype}")
    else:
        print(f"  {field:10s}  ->  {val}")

TranMatFI namedtuple created with fields:
  Pi          ->  shape (941, 941), dtype float64
  S           ->  shape (941, 3), dtype float64
  Yn          ->  941
  yt          ->  shape (10,), dtype float64
  yn          ->  shape (10,), dtype float64
  gt          ->  shape (10,), dtype float64
  Xsim        ->  shape (2000000, 3), dtype float64
  Xsim_idx    ->  shape (2000000,), dtype int64


### Verification

Quick sanity checks on the transition matrix.

In [45]:
# Verify transition matrix properties
row_sums = Pi.sum(axis=1)
print(f"Row sums — min: {row_sums.min():.10f},  max: {row_sums.max():.10f}")
print(f"All entries non-negative: {bool(jnp.all(Pi >= 0))}")
print(f"Pi is square: {Pi.shape[0] == Pi.shape[1]}")
print(f"States consistent with Pi: {S.shape[0] == Pi.shape[0]}")

Row sums — min: 1.0000000000,  max: 1.0000000000
All entries non-negative: True
Pi is square: True
States consistent with Pi: True


In [56]:
# Print the first 10 values of the simulated state variables
print("\nFirst 10 simulated state values (yt, yn, gt):")
for i in range(10):
    print(f"  t={i:3d}: yt={S[i, 0]:.6f}, yn={S[i, 1]:.6f}, gt={S[i, 2]:.6f}") 
    
print(yt)



First 10 simulated state values (yt, yn, gt):
  t=  0: yt=-0.312454, yn=-0.130176, gt=-0.102831
  t=  1: yt=-0.312454, yn=-0.130176, gt=-0.056493
  t=  2: yt=-0.312454, yn=-0.130176, gt=-0.010154
  t=  3: yt=-0.312454, yn=-0.130176, gt=0.036184
  t=  4: yt=-0.312454, yn=-0.130176, gt=0.082523
  t=  5: yt=-0.312454, yn=-0.130176, gt=0.128861
  t=  6: yt=-0.312454, yn=-0.101248, gt=-0.149170
  t=  7: yt=-0.312454, yn=-0.101248, gt=-0.102831
  t=  8: yt=-0.312454, yn=-0.101248, gt=-0.056493
  t=  9: yt=-0.312454, yn=-0.101248, gt=-0.010154
[-0.31245414 -0.24301989 -0.17358563 -0.10415138 -0.03471713  0.03471713
  0.10415138  0.17358563  0.24301989  0.31245414]
